In [1]:
import os
os.listdir("/content")

['.config', 'sample_data']

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import explode

In [3]:
spark = SparkSession.builder \
    .appName("Movie Recommendation ALS") \
    .getOrCreate()

print("Spark is ready")

Spark is ready


In [4]:
movies = spark.read.csv("/content/clean_movies.csv", header=True, inferSchema=True)
ratings = spark.read.csv("/content/clean_ratings.csv", header=True, inferSchema=True)

movies.show(5)
ratings.show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|    169|Free Willy 2: The...|Adventure|Childre...|
|    191|Scarlet Letter, T...|               Drama|
|    891|Halloween: The Cu...|     Horror|Thriller|
|   1330|April Fool's Day ...|       Comedy|Horror|
|   1381|     Grease 2 (1982)|Comedy|Musical|Ro...|
+-------+--------------------+--------------------+
only showing top 5 rows
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     5|   1966|     2|978245931|
|     8|   3147|     5|978230550|
|    10|   2396|     5|979167660|
|    11|   1394|     4|978902965|
|    11|   3174|     2|978903658|
+------+-------+------+---------+
only showing top 5 rows


In [5]:
ratings.printSchema()
movies.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [6]:
ratings = ratings.select("userId", "movieId", "rating")

In [7]:
train_data, test_data = ratings.randomSplit([0.8, 0.2], seed=42)

print("Train count:", train_data.count())
print("Test count:", test_data.count())

Train count: 396894
Test count: 99225


In [8]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",   # IMPORTANT (prevents NaN issue)
    nonnegative=True
)

In [9]:
model = als.fit(train_data)

In [10]:
predictions = model.transform(test_data)

predictions.show(5)

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|   148|    107|     4|  3.286327|
|   148|    173|     3| 2.9077013|
|   148|    239|     4|   3.48947|
|   148|    253|     3| 3.7404122|
|   148|    277|     4| 3.6493902|
+------+-------+------+----------+
only showing top 5 rows


In [11]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)

print("RMSE =", rmse)

RMSE = 0.8966639458193306


In [12]:
user_recommendations = model.recommendForAllUsers(10)

user_recommendations.show(5, truncate=False)

+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                           |
+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1     |[{3077, 5.1143613}, {2309, 5.0828056}, {666, 5.0227094}, {53, 4.9535437}, {3222, 4.91434}, {3517, 4.8657193}, {2197, 4.7997847}, {649, 4.7284846}, {2823, 4.7196836}, {213, 4.7087727}]   |
|3     |[{864, 4.5825715}, {649, 4.5305023}, {2324, 4.49471}, {3147, 4.4620523}, {318, 4.4602456}, {2309, 4.419184}, {590, 4.3805323}, {863, 4.3765473}, {527, 4.3649597}, {3753, 4.3614326}]     |
|5     |[{850, 4.912

In [13]:
user_subset = ratings.select("userId").distinct().limit(1)

single_user_recs = model.recommendForUserSubset(user_subset, 10)

single_user_recs.show(truncate=False)

+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                           |
+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|5     |[{850, 4.9128084}, {2981, 4.6971607}, {2931, 4.5926914}, {3429, 4.3997965}, {2823, 4.3951955}, {759, 4.3888755}, {2309, 4.3715267}, {3569, 4.3347683}, {3150, 4.331361}, {670, 4.2888107}]|
+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [15]:
from pyspark.sql.functions import col

final_recs = user_recommendations \
    .withColumn("rec", explode("recommendations")) \
    .select(
        col("userId"),
        col("rec.movieId").alias("movieId"),
        col("rec.rating").alias("rec_rating")
    )

final_recs = final_recs.join(movies, on="movieId", how="inner")

final_recs.show(10, truncate=False)

+-------+------+----------+-----------------------------------------------------------+------------+
|movieId|userId|rec_rating|title                                                      |genres      |
+-------+------+----------+-----------------------------------------------------------+------------+
|3077   |1     |5.1143613 |42 Up (1998)                                               |Documentary |
|2309   |1     |5.0828056 |Inheritors, The (Die Siebtelbauern) (1998)                 |Drama       |
|666    |1     |5.0227094 |All Things Fair (1996)                                     |Drama       |
|53     |1     |4.9535437 |Lamerica (1994)                                            |Drama       |
|3222   |1     |4.91434   |Carmen (1984)                                              |Drama       |
|3517   |1     |4.8657193 |Bells, The (1926)                                          |Crime|Drama |
|2197   |1     |4.7997847 |Firelight (1997)                                           |Dram

In [16]:
final_recs.write.csv("/content/recommendations_output", header=True)